# Immune-evasion and stemness gene scores in breast cancer: a survival analysis

**Author:** Dickens Agie, Biochemist, Nairobi, Kenya

**Question.** Do breast tumours that score high on an *immune-evasion* gene module, or on a *stemness* gene module,
have different survival from tumours that score low, and does the pattern seen in TCGA-BRCA hold up in the
independent METABRIC cohort?

**How to run this notebook (Google Colab, works from a phone browser).**
1. Open the notebook in Colab and choose *Runtime → Run all*.
2. The first run downloads the two public cohorts from the cBioPortal data hub (a few hundred MB) and takes roughly 10-20 minutes.
3. Tables are written to `results/` and figures to `figures/`. The last cell zips them so you can download them.

**Please read.** The setting `USE_DEMO_DATA` below switches the notebook to *simulated* data so the code can be tested
without a download. Simulated runs are written to `demo_outputs/`, carry a red watermark on every figure, and must never
be quoted as findings. All real results come from `USE_DEMO_DATA = False`.

## 1. Setup

In [ ]:
import sys, subprocess, importlib


def _ensure(pkg, import_name=None):
    """Install a package only if it is missing (Colab already has most of them)."""
    try:
        importlib.import_module(import_name or pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])


_ensure("lifelines")

import os, re, json, hashlib, tarfile, shutil, platform, warnings, urllib.request
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import scipy
import lifelines
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import logrank_test, proportional_hazard_test
from lifelines.plotting import add_at_risk_counts

warnings.filterwarnings("ignore", category=FutureWarning)

## 2. Settings and the pre-specified gene signature

The signature is fixed here, **before** any survival result is looked at, and it is never tuned afterwards.
Genes are not weighted or fitted to outcome: each module score is the plain average of z-scored expression.
That means there is no training step that could overfit, and the METABRIC run is a genuine replication of the
same fixed rule.

| Module | Genes | Why these genes |
|---|---|---|
| Immune evasion | CD274 (PD-L1), PDCD1LG2 (PD-L2), CTLA4, LAG3, TIGIT, IDO1, CD47 | Checkpoint ligands and receptors, an immunosuppressive enzyme (IDO1) and the "don't eat me" signal (CD47) that tumours use to dampen or avoid immune attack |
| Stemness | ALDH1A1, CD44, PROM1 (CD133), SOX2 | Markers commonly used to describe cancer stem-like cells in breast cancer |

**An important caution about interpretation.** Several checkpoint genes (CTLA4, LAG3, TIGIT, PDCD1LG2) are mostly
expressed by immune cells that have infiltrated the tumour. A high immune-evasion score can therefore mean
"a heavily inflamed tumour", which in some breast cancer subtypes goes with *better* outcome, not worse.
This is why the two modules are analysed separately and the direction of effect is not assumed in advance.

In [ ]:
# ---- Analysis settings -----------------------------------------------------
USE_DEMO_DATA = False      # True = test the code on SIMULATED data (never use for results)
MAX_MONTHS = 120           # follow-up is censored at 10 years in both cohorts
MIN_EVENTS_SUBTYPE = 20    # smallest number of events needed to fit a subtype-specific model
RANDOM_SEED = 2026

# ---- The signature (fixed in advance) --------------------------------------
SIGNATURE = {
    "immune_evasion": ["CD274", "PDCD1LG2", "CTLA4", "LAG3", "TIGIT", "IDO1", "CD47"],
    "stemness": ["ALDH1A1", "CD44", "PROM1", "SOX2"],
}
MIN_GENES_PER_MODULE = 3   # if a cohort lacks too many genes, stop instead of quietly changing the signature
ALL_SIGNATURE_GENES = [g for genes in SIGNATURE.values() for g in genes]
MODULES = ["immune_evasion", "stemness", "combined"]   # 'combined' = mean of the two module scores (exploratory)
PRIMARY_MODULES = ["immune_evasion", "stemness"]
PRIMARY_ENDPOINT = "OS"

MODULE_LABELS = {"immune_evasion": "Immune-evasion", "stemness": "Stemness", "combined": "Combined"}
ENDPOINT_LABELS = {"OS": "overall survival", "PFS": "progression-free survival",
                   "RFS": "relapse-free survival", "DSS": "disease-specific survival"}

# ---- Cohorts ---------------------------------------------------------------
# Both studies are public and de-identified (cBioPortal data hub).
# 'endpoints' maps a name to (time column, event column, how the event is coded).
COHORTS = {
    "TCGA-BRCA": dict(
        study_id="brca_tcga_pan_can_atlas_2018",
        expression_preferred="data_mrna_seq_v2_rsem.txt",
        primary_tumour_suffix="-01",
        endpoints={"OS": ("OS_MONTHS", "OS_STATUS", "cbio"),
                   "PFS": ("PFS_MONTHS", "PFS_STATUS", "cbio")},
        overrides={},       # e.g. {"subtype": "SUBTYPE"} if a column name differs
    ),
    "METABRIC": dict(
        study_id="brca_metabric",
        expression_preferred="data_mrna_illumina_microarray.txt",
        primary_tumour_suffix=None,
        endpoints={"OS": ("OS_MONTHS", "OS_STATUS", "cbio"),
                   "RFS": ("RFS_MONTHS", "RFS_STATUS", "cbio"),
                   "DSS": ("OS_MONTHS", "VITAL_STATUS", "vital")},
        overrides={},
    ),
}

# Column names differ between studies; the first match in each list is used.
COLUMN_CANDIDATES = {
    "age": ["AGE_AT_DIAGNOSIS", "AGE", "DIAGNOSIS_AGE"],
    "stage": ["AJCC_PATHOLOGIC_TUMOR_STAGE", "TUMOR_STAGE", "STAGE"],
    "subtype": ["SUBTYPE", "CLAUDIN_SUBTYPE", "PAM50_SUBTYPE", "PAM50"],
    "sex": ["SEX", "GENDER"],
}

DATAHUB_URL = "https://cbioportal-datahub.s3.amazonaws.com/{study_id}.tar.gz"

# ---- Folders ---------------------------------------------------------------
OUT_DIR = Path("demo_outputs") if USE_DEMO_DATA else Path(".")
RAW_DIR = Path("data/demo") if USE_DEMO_DATA else Path("data/raw")
RESULTS_DIR = OUT_DIR / "results"
FIG_DIR = OUT_DIR / "figures"
for _d in (RAW_DIR, RESULTS_DIR, FIG_DIR):
    _d.mkdir(parents=True, exist_ok=True)

## 3. Helper functions: reading cBioPortal files

In [ ]:
NA_VALUES = ["", "NA", "N/A", "[Not Available]", "[Not Applicable]", "[Unknown]",
             "[Not Evaluated]", "[Pending]", "Not Available", "null", "NaN"]


def pick_col(df, candidates, what, required=True):
    """Return the first column of df whose name matches one of the candidates (case-insensitive)."""
    lookup = {str(c).lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lookup:
            return lookup[cand.lower()]
    if required:
        raise KeyError(f"No column found for '{what}'. Tried {candidates}. "
                       f"Columns present: {list(df.columns)}")
    return None


def read_cbio_table(path):
    """cBioPortal clinical files start with a few '#' description lines; skip them."""
    skip = 0
    with open(path, "r", encoding="utf-8", errors="replace") as fh:
        for line in fh:
            if line.startswith("#"):
                skip += 1
            else:
                break
    return pd.read_csv(path, sep="\t", skiprows=skip, dtype=str,
                       na_values=NA_VALUES, low_memory=False)


def sha256_of(path, block=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(block), b""):
            h.update(chunk)
    return h.hexdigest()


def find_expression_file(folder, preferred):
    folder = Path(folder)
    if (folder / preferred).exists():
        return folder / preferred
    options = sorted(p for p in folder.glob("data_mrna*.txt") if "zscore" not in p.name.lower())
    if len(options) == 1:
        return options[0]
    raise FileNotFoundError(
        f"Expected '{preferred}' in {folder}. mRNA files found: {[p.name for p in options]}. "
        "Set 'expression_preferred' for this cohort in COHORTS to the right file name.")


def read_expression_for_genes(path, genes):
    """Read a (genes x samples) cBioPortal expression file, keeping only the rows we need."""
    wanted = set(genes)
    kept, symbol_col = [], None
    for chunk in pd.read_csv(path, sep="\t", chunksize=2000, low_memory=False):
        if symbol_col is None:
            symbol_col = pick_col(chunk, ["Hugo_Symbol", "gene", "symbol"], "gene symbol")
        sub = chunk[chunk[symbol_col].isin(wanted)]
        if len(sub):
            kept.append(sub)
    if not kept:
        raise ValueError(f"None of the signature genes were found in {path}")
    df = pd.concat(kept, axis=0)
    df = df.drop(columns=[c for c in df.columns
                          if str(c).lower() in ("entrez_gene_id", "entrez", "description", "gene_id")])
    df = df.set_index(symbol_col)
    df = df.apply(pd.to_numeric, errors="coerce")
    return df.groupby(level=0).mean()          # if a gene has several rows, average them


def maybe_log2(expr):
    """RSEM counts are on a linear scale (use log2(x+1)); microarray values are already log2."""
    top = np.nanmax(expr.to_numpy(dtype=float))
    if top > 100:
        return np.log2(expr + 1), True
    return expr, False

## 4. Downloading the cohorts (or simulating data when `USE_DEMO_DATA = True`)

If the automatic download ever fails, download each study from cBioPortal (study page → *Download*),
unzip it, and place `data_clinical_patient.txt`, `data_clinical_sample.txt` and the mRNA expression file
in `data/raw/<study_id>/`. The notebook will use them.

In [ ]:
def download_study_files(study_id, dest_dir):
    dest_dir = Path(dest_dir)
    dest_dir.mkdir(parents=True, exist_ok=True)
    needed = {"data_clinical_patient.txt", "data_clinical_sample.txt"}
    if all((dest_dir / f).exists() for f in needed) and any(dest_dir.glob("data_mrna*.txt")):
        print(f"  {study_id}: files already present, skipping download")
        return
    tar_path = dest_dir / f"{study_id}.tar.gz"
    if not tar_path.exists():
        url = DATAHUB_URL.format(study_id=study_id)
        print(f"  downloading {url}")
        with urllib.request.urlopen(url, timeout=120) as resp, open(tar_path, "wb") as out:
            shutil.copyfileobj(resp, out, length=1 << 20)
    print("  extracting the clinical and mRNA files")
    with tarfile.open(tar_path, "r:gz") as tf:
        for member in tf:
            if not member.isfile():
                continue
            parts = Path(member.name).parts
            if ".." in parts or Path(member.name).is_absolute():
                continue                       # never write outside the target folder
            name = Path(member.name).name
            keep = name in needed or (name.startswith("data_mrna") and "zscore" not in name.lower())
            if keep:
                with tf.extractfile(member) as src, open(dest_dir / name, "wb") as dst:
                    shutil.copyfileobj(src, dst, length=1 << 20)
    tar_path.unlink()
    if not all((dest_dir / f).exists() for f in needed):
        raise FileNotFoundError(f"Clinical files missing after extraction in {dest_dir}. "
                                "Use the manual download described above.")


def simulate_cohort_files(name, folder):
    """SIMULATED data in cBioPortal file layout, only for testing the code. Not real patients."""
    rng = np.random.default_rng(RANDOM_SEED + (0 if name == "TCGA-BRCA" else 1))
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    is_tcga = name == "TCGA-BRCA"
    n = 900 if is_tcga else 1700
    patient = [f"{'TCGA-AA' if is_tcga else 'MB'}-{i:04d}" for i in range(1, n + 1)]
    sample = [p + "-01" for p in patient] if is_tcga else list(patient)

    subtype = rng.choice(["LumA", "LumB", "Her2", "Basal", "Normal"], size=n, p=[0.45, 0.22, 0.10, 0.18, 0.05])
    shift_imm = {"LumA": -0.3, "LumB": 0.0, "Her2": 0.4, "Basal": 0.6, "Normal": -0.2}
    shift_stem = {"LumA": -0.3, "LumB": 0.0, "Her2": 0.0, "Basal": 0.7, "Normal": 0.1}
    imm = rng.normal(0, 1, n) + np.array([shift_imm[s] for s in subtype])
    stem = rng.normal(0, 1, n) + np.array([shift_stem[s] for s in subtype])
    age = np.clip(rng.normal(58, 12, n), 25, 90).round(0)
    stage = rng.choice([1, 2, 3, 4], size=n, p=[0.2, 0.5, 0.25, 0.05])

    # Built-in truth: stemness raises hazard (HR 1.30 per latent SD), immune activity lowers it (0.85).
    lin = (np.log(1.30) * stem + np.log(0.85) * imm + 0.35 * (stage - 2)
           + 0.02 * (age - 58) + 0.3 * (subtype == "Basal"))
    base = 0.0012 if is_tcga else 0.0035
    t_death = rng.exponential(1 / (base * np.exp(lin)))
    t_prog = rng.exponential(1 / (1.8 * base * np.exp(lin)))
    cens = rng.exponential(40, n) if is_tcga else rng.uniform(20, 300, n)
    os_t = np.minimum(t_death, cens).round(2)
    os_e = (t_death <= cens).astype(int)
    pfs_t = np.minimum.reduce([t_prog, t_death, cens]).round(2)
    pfs_e = ((t_prog <= cens) | (t_death <= cens)).astype(int)
    other_cause = (rng.random(n) < 0.3) & (os_e == 1)

    # ---- clinical files (with cBioPortal-style '#' header lines)
    def write_clinical(path, df):
        cols = list(df.columns)
        with open(path, "w") as fh:
            for _ in range(4):
                fh.write("#" + "\t".join(["header"] * len(cols)) + "\n")
            fh.write("\t".join(cols) + "\n")
            for row in df.itertuples(index=False):
                fh.write("\t".join("" if (isinstance(v, float) and np.isnan(v)) else str(v) for v in row) + "\n")

    if is_tcga:
        roman = {1: "STAGE I", 2: "STAGE II", 3: "STAGE III", 4: "STAGE IV"}
        stage_txt = [roman[s] + (rng.choice(["", "A", "B"]) if s < 4 else "") for s in stage]
        for i in rng.choice(n, 36, replace=False):
            stage_txt[i] = "[Not Available]" if i % 2 else "STAGE X"
        sex = np.array(["Female"] * n, dtype=object)
        sex[rng.choice(n, 8, replace=False)] = "Male"
        pat = pd.DataFrame({
            "PATIENT_ID": patient, "SEX": sex, "AGE": age.astype(int),
            "AJCC_PATHOLOGIC_TUMOR_STAGE": stage_txt,
            "SUBTYPE": ["BRCA_" + s for s in subtype],
            "OS_STATUS": np.where(os_e == 1, "1:DECEASED", "0:LIVING"), "OS_MONTHS": os_t,
            "PFS_STATUS": np.where(pfs_e == 1, "1:PROGRESSION", "0:CENSORED"), "PFS_MONTHS": pfs_t})
        extra = list(rng.choice(n, 20, replace=False))              # metastatic duplicates ('-06')
        smp = pd.DataFrame({"PATIENT_ID": patient + [patient[i] for i in extra],
                            "SAMPLE_ID": sample + [patient[i] + "-06" for i in extra]})
        sample_all = list(smp["SAMPLE_ID"])
    else:
        vital = np.where(os_e == 0, "Living", np.where(other_cause, "Died of Other Causes", "Died of Disease"))
        stage_num = stage.astype(float)
        stage_num[rng.choice(n, 60, replace=False)] = np.nan
        pat = pd.DataFrame({
            "PATIENT_ID": patient, "AGE_AT_DIAGNOSIS": age, "TUMOR_STAGE": stage_num,
            "CLAUDIN_SUBTYPE": subtype,
            "OS_MONTHS": os_t, "OS_STATUS": np.where(os_e == 1, "1:DECEASED", "0:LIVING"),
            "VITAL_STATUS": vital,
            "RFS_MONTHS": pfs_t, "RFS_STATUS": np.where(pfs_e == 1, "1:Recurred", "0:Not Recurred")})
        smp = pd.DataFrame({"PATIENT_ID": patient, "SAMPLE_ID": sample})
        sample_all = list(sample)
    write_clinical(folder / "data_clinical_patient.txt", pat)
    write_clinical(folder / "data_clinical_sample.txt", smp)

    # ---- expression file
    genes = list(ALL_SIGNATURE_GENES) + [f"GENE{i:03d}" for i in range(1, 61)]
    if not is_tcga:
        genes.remove("PROM1")           # deliberately absent so the 'common genes only' rule is exercised
    index_of = {s: i for i, s in enumerate(sample)}
    latent_i = np.array([imm[index_of[s.rsplit("-", 1)[0] + "-01"]] if is_tcga else imm[index_of[s]] for s in sample_all])
    latent_s = np.array([stem[index_of[s.rsplit("-", 1)[0] + "-01"]] if is_tcga else stem[index_of[s]] for s in sample_all])
    rows = []
    for g in genes:
        mu = rng.uniform(4, 10)
        if g in SIGNATURE["immune_evasion"]:
            x = mu + 0.9 * latent_i + rng.normal(0, 0.6, len(sample_all))
        elif g in SIGNATURE["stemness"]:
            x = mu + 0.8 * latent_s + rng.normal(0, 0.7, len(sample_all))
        else:
            x = mu + rng.normal(0, 1.0, len(sample_all))
        rows.append((g, x))
    rows.append(("CD44", rows[[r[0] for r in rows].index("CD44")][1] + rng.normal(0, 0.05, len(sample_all))))  # duplicate row
    fname = "data_mrna_seq_v2_rsem.txt" if is_tcga else "data_mrna_illumina_microarray.txt"
    with open(folder / fname, "w") as fh:
        fh.write("Hugo_Symbol\tEntrez_Gene_Id\t" + "\t".join(sample_all) + "\n")
        for k, (g, x) in enumerate(rows):
            vals = (2 ** x - 1) if is_tcga else x         # RSEM-like linear scale vs log2 microarray scale
            cells = ["NA" if (j % 997 == 13 and k == 3) else f"{v:.4f}" for j, v in enumerate(vals)]
            fh.write(f"{g}\t{1000 + k}\t" + "\t".join(cells) + "\n")

## 5. Building one tidy table per cohort

Every filtering step is counted and saved to `results/sample_flow.csv`, so the final sample sizes can be explained.

In [ ]:
def stage_to_num(value):
    """'STAGE IIA' -> 2, '3' -> 3, anything unusable -> NaN."""
    if pd.isna(value):
        return np.nan
    text = str(value).strip().upper()
    try:
        num = float(text)
        return num if 1 <= num <= 4 else np.nan
    except ValueError:
        pass
    match = re.search(r"STAGE\s*(IV|III|II|I)", text)       # 'STAGE IIA' -> II
    return {"I": 1, "II": 2, "III": 3, "IV": 4}[match.group(1)] if match else np.nan


def norm_subtype(value):
    if pd.isna(value):
        return np.nan
    t = str(value).lower().replace("brca_", "").replace(" ", "")
    mapping = {"luma": "LumA", "lumb": "LumB", "her2": "Her2", "her2-enriched": "Her2",
               "her2enriched": "Her2", "basal": "Basal", "basal-like": "Basal", "normal": "Normal",
               "normal-like": "Normal", "claudin-low": "Claudin-low", "claudinlow": "Claudin-low"}
    return mapping.get(t, np.nan)


def parse_event(series, kind):
    """Turn cBioPortal event strings into 1 (event) / 0 (censored) / NaN."""
    s = series.astype(str).str.strip()
    if kind == "cbio":                          # '1:DECEASED', '0:LIVING', '1:Recurred', ...
        out = pd.to_numeric(s.str.split(":").str[0], errors="coerce")
    elif kind == "vital":                       # disease-specific: other-cause deaths are censored
        low = s.str.lower()
        out = pd.Series(np.nan, index=series.index)
        out[low.str.contains("died of disease", na=False)] = 1.0
        out[low.str.contains("living", na=False)] = 0.0
        out[low.str.contains("died of other", na=False)] = 0.0
    else:
        raise ValueError(kind)
    return out.where(out.isin([0, 1]))


def build_cohort(name, cfg):
    print(f"== {name}")
    folder = RAW_DIR / cfg["study_id"]
    if USE_DEMO_DATA:
        simulate_cohort_files(name, folder)
    else:
        download_study_files(cfg["study_id"], folder)

    pat = read_cbio_table(folder / "data_clinical_patient.txt")
    smp = read_cbio_table(folder / "data_clinical_sample.txt")
    expr_path = find_expression_file(folder, cfg["expression_preferred"])
    expr = read_expression_for_genes(expr_path, ALL_SIGNATURE_GENES)
    expr, was_logged = maybe_log2(expr)

    clin = smp.merge(pat, on="PATIENT_ID", how="left", suffixes=("", "_pt"))
    flow = [("Samples listed in the clinical file", len(clin))]

    clin = clin[clin["SAMPLE_ID"].isin(expr.columns)]
    flow.append(("... with expression data", len(clin)))

    suffix = cfg["primary_tumour_suffix"]
    if suffix:
        mask = clin["SAMPLE_ID"].str.endswith(suffix)
        if mask.sum() == 0:
            print(f"  WARNING: no sample IDs end in '{suffix}'; keeping all samples")
        else:
            clin = clin[mask]
        flow.append(("... primary tumour samples only", len(clin)))

    clin = clin.drop_duplicates("PATIENT_ID", keep="first")
    flow.append(("... one sample per patient", len(clin)))

    sex_col = pick_col(clin, [cfg["overrides"].get("sex")] if cfg["overrides"].get("sex") else COLUMN_CANDIDATES["sex"],
                       "sex", required=False)
    if sex_col is not None:
        is_male = clin[sex_col].astype(str).str.lower().str.startswith("m")
        clin = clin[~is_male]
        flow.append(("... women only", len(clin)))

    clin = clin.set_index("SAMPLE_ID")
    tidy = pd.DataFrame(index=clin.index)
    tidy["patient"] = clin["PATIENT_ID"]

    def choose(key):
        override = cfg["overrides"].get(key)
        return pick_col(clin, [override] if override else COLUMN_CANDIDATES[key], key, required=False)

    age_col = choose("age")
    tidy["age"] = pd.to_numeric(clin[age_col], errors="coerce") if age_col else np.nan
    stage_col = choose("stage")
    tidy["stage"] = clin[stage_col].map(stage_to_num) if stage_col else np.nan

    tidy["subtype"] = np.nan
    subtype_options = [cfg["overrides"]["subtype"]] if cfg["overrides"].get("subtype") else COLUMN_CANDIDATES["subtype"]
    for cand in subtype_options:
        col = pick_col(clin, [cand], "subtype", required=False)
        if col is None:
            continue
        mapped = clin[col].map(norm_subtype)
        if mapped.notna().mean() >= 0.5:          # use the first column that actually looks like a PAM50-type call
            tidy["subtype"] = mapped
            print(f"  subtype taken from column '{col}'")
            break

    endpoints_found = []
    for ename, (tcol, ecol, kind) in cfg["endpoints"].items():
        t_col = pick_col(clin, [tcol], f"{ename} time", required=False)
        e_col = pick_col(clin, [ecol], f"{ename} event", required=False)
        if t_col is None or e_col is None:
            print(f"  note: endpoint {ename} skipped (columns {tcol}/{ecol} not found)")
            continue
        time = pd.to_numeric(clin[t_col], errors="coerce")
        event = parse_event(clin[e_col], kind)
        bad = time.isna() | (time <= 0) | event.isna()
        time, event = time.where(~bad), event.where(~bad)
        over = time > MAX_MONTHS                  # administrative censoring at MAX_MONTHS
        time = time.where(~over, MAX_MONTHS)
        event = event.where(~over, 0)
        tidy[f"{ename}_time"], tidy[f"{ename}_event"] = time, event
        endpoints_found.append(ename)
    if PRIMARY_ENDPOINT not in endpoints_found:
        raise KeyError(f"Primary endpoint {PRIMARY_ENDPOINT} could not be built for {name}")

    expr_t = expr.T.loc[tidy.index]
    flow.append((f"Final cohort ({PRIMARY_ENDPOINT} available: "
                 f"{int(tidy[f'{PRIMARY_ENDPOINT}_time'].notna().sum())})", len(tidy)))
    meta = dict(study_id=cfg["study_id"], expression_file=expr_path.name, expression_was_log_transformed=was_logged,
                genes_found=sorted(expr.index), genes_missing=sorted(set(ALL_SIGNATURE_GENES) - set(expr.index)),
                sha256={p.name: sha256_of(p) for p in (folder / "data_clinical_patient.txt",
                                                       folder / "data_clinical_sample.txt", expr_path)})
    print(f"  {len(tidy)} patients, endpoints: {endpoints_found}, genes found: {len(expr.index)}/{len(ALL_SIGNATURE_GENES)}")
    return dict(tidy=tidy, expr=expr_t, endpoints=endpoints_found, flow=flow, meta=meta)


def finalise_signature(cohorts):
    """Use only genes present in every cohort, so the score means the same thing in both."""
    rows, final = [], {}
    for module, genes in SIGNATURE.items():
        keep = []
        for g in genes:
            present = {n: (g in c["expr"].columns) for n, c in cohorts.items()}
            rows.append({"module": module, "gene": g, **{f"in_{n}": v for n, v in present.items()},
                         "used": all(present.values())})
            if all(present.values()):
                keep.append(g)
        if len(keep) < MIN_GENES_PER_MODULE:
            raise ValueError(f"Module '{module}' has only {len(keep)} genes present in all cohorts "
                             f"(minimum {MIN_GENES_PER_MODULE}). Not changing the signature automatically.")
        final[module] = keep
    return final, pd.DataFrame(rows)

## 6. Signature scores

For each cohort separately: expression is put on a log2 scale, each gene is converted to a z-score across patients
in that cohort, the module score is the mean of its genes' z-scores, and the score itself is re-standardised so that
hazard ratios are "per 1 standard deviation of the score". Standardising inside each cohort removes the platform
difference between RNA-seq (TCGA) and microarray (METABRIC); it does not use any outcome information.

In [ ]:
def zscore_cols(df):
    return (df - df.mean()) / df.std(ddof=0)


def compute_scores(expr_log, signature_final):
    genes = [g for gs in signature_final.values() for g in gs]
    z = zscore_cols(expr_log[genes])
    scores = pd.DataFrame({m: z[gs].mean(axis=1) for m, gs in signature_final.items()})
    scores["combined"] = scores[list(signature_final)].mean(axis=1)
    return zscore_cols(scores), z

## 7. Survival analysis

For every cohort, endpoint and module the notebook reports:
* a **median split** (high vs low), log-rank test and hazard ratio, for the Kaplan-Meier curves;
* a **continuous Cox model**, hazard ratio per 1 SD of the score (the primary, pre-specified analysis);
* an **adjusted Cox model** (age, stage and PAM50-type subtype, whichever are available for at least 70-80% of patients),
  with a proportional-hazards check and Harrell's C-index.

**Pre-specified plan.** Primary question: overall survival, the two module scores, continuous hazard ratio, TCGA-BRCA as the
discovery cohort. A module is called *replicated* if, in METABRIC, the effect has the same direction and the 95% CI excludes 1.
Everything else (the combined score, secondary endpoints, subtypes) is exploratory and labelled that way.
Benjamini-Hochberg adjusted p-values are provided across all tests.

In [ ]:
def fmt_p(p):
    if pd.isna(p):
        return "NA"
    return f"{p:.3f}" if p >= 0.001 else f"{p:.1e}"


def benjamini_hochberg(p):
    p = np.asarray(p, dtype=float)
    out = np.full_like(p, np.nan)
    idx = np.where(~np.isnan(p))[0]
    if len(idx) == 0:
        return out
    order = idx[np.argsort(p[idx])]
    ranked = p[order] * len(idx) / np.arange(1, len(idx) + 1)
    ranked = np.minimum.accumulate(ranked[::-1])[::-1]
    out[order] = np.minimum(ranked, 1.0)
    return out


def endpoint_frame(tidy, scores, endpoint):
    df = pd.concat([tidy, scores], axis=1)
    df = df.rename(columns={f"{endpoint}_time": "time", f"{endpoint}_event": "event"})
    df = df[["time", "event", "age", "stage", "subtype"] + MODULES]
    return df.dropna(subset=["time", "event"])


def fit_cox(data, var):
    cph = CoxPHFitter()
    cph.fit(data, duration_col="time", event_col="event")
    s = cph.summary.loc[var]
    return cph, dict(hr=float(s["exp(coef)"]), lo=float(s["exp(coef) lower 95%"]),
                     hi=float(s["exp(coef) upper 95%"]), p=float(s["p"]))


def choose_covariates(df):
    chosen = []
    if df["age"].notna().mean() >= 0.8:
        chosen.append("age")
    if df["stage"].notna().mean() >= 0.7:
        chosen.append("stage")
    if df["subtype"].notna().mean() >= 0.7:
        chosen.append("subtype")
    return chosen


def design_matrix(df, covariates):
    X = pd.DataFrame(index=df.index)
    if "age" in covariates:
        X["age_per10y"] = df["age"] / 10
    if "stage" in covariates:
        X["stage"] = df["stage"]
    if "subtype" in covariates:
        sub = df["subtype"]
        for level in sorted(sub.dropna().unique()):
            if level != "LumA":                               # LumA is the reference group
                X[f"subtype_{level}"] = (sub == level).astype(float).where(sub.notna())
    return X


def analyse_frame(cohort, endpoint, df):
    rows, covs = [], choose_covariates(df)
    for module in MODULES:
        d = df[["time", "event", module]].dropna()
        row = dict(cohort=cohort, endpoint=endpoint, module=module, n=len(d), events=int(d["event"].sum()),
                   primary=(endpoint == PRIMARY_ENDPOINT and module in PRIMARY_MODULES))
        try:
            high = d[module] > d[module].median()
            lr = logrank_test(d.loc[high, "time"], d.loc[~high, "time"], d.loc[high, "event"], d.loc[~high, "event"])
            _, r = fit_cox(d.assign(high=high.astype(float))[["time", "event", "high"]], "high")
            row.update(logrank_p_median_split=lr.p_value, hr_high_vs_low=r["hr"],
                       hr_high_lo=r["lo"], hr_high_hi=r["hi"])
            cph, r = fit_cox(d, module)
            row.update(hr_per_sd=r["hr"], ci_lo=r["lo"], ci_hi=r["hi"], p=r["p"])
            try:
                row["ph_test_p_unadjusted"] = float(
                    proportional_hazard_test(cph, d, time_transform="rank").summary.loc[module, "p"])
            except Exception:
                row["ph_test_p_unadjusted"] = np.nan
        except Exception as exc:
            row["note"] = f"unadjusted model failed: {exc}"
        if covs:
            try:
                X = design_matrix(df, covs)
                data = pd.concat([df[["time", "event", module]], X], axis=1).dropna()
                cph, r = fit_cox(data, module)
                row.update(adj_hr=r["hr"], adj_lo=r["lo"], adj_hi=r["hi"], adj_p=r["p"],
                           adj_n=len(data), adj_events=int(data["event"].sum()),
                           adj_c_index=float(cph.concordance_index_), adjusted_for=", ".join(covs))
                try:
                    row["ph_test_p_adjusted"] = float(
                        proportional_hazard_test(cph, data, time_transform="rank").summary.loc[module, "p"])
                except Exception:
                    row["ph_test_p_adjusted"] = np.nan
            except Exception as exc:
                row["note"] = (row.get("note", "") + f" adjusted model failed: {exc}").strip()
        rows.append(row)
    return rows


def analyse_subtypes(cohort, endpoint, df):
    rows = []
    for st in ["LumA", "LumB", "Her2", "Basal"]:
        sub = df[df["subtype"] == st]
        for module in PRIMARY_MODULES:
            d = sub[["time", "event", "age", module]].dropna(subset=["time", "event", module])
            row = dict(cohort=cohort, endpoint=endpoint, subtype=st, module=module,
                       n=len(d), events=int(d["event"].sum()))
            if row["events"] < MIN_EVENTS_SUBTYPE:
                row["note"] = f"fewer than {MIN_EVENTS_SUBTYPE} events: not modelled"
            else:
                try:
                    data = d[["time", "event", module]].copy()
                    if d["age"].notna().mean() >= 0.8:
                        data["age_per10y"] = d["age"] / 10
                    data = data.dropna()
                    _, r = fit_cox(data, module)
                    row.update(hr_per_sd=r["hr"], ci_lo=r["lo"], ci_hi=r["hi"], p=r["p"])
                except Exception as exc:
                    row["note"] = f"model failed: {exc}"
            rows.append(row)
    return rows


def replication_table(results):
    """Primary question: same direction in METABRIC as in TCGA-BRCA, and does the METABRIC CI exclude 1?"""
    out = []
    prim = results[(results["endpoint"] == PRIMARY_ENDPOINT) & (results["module"].isin(PRIMARY_MODULES))]
    for module in PRIMARY_MODULES:
        a = prim[(prim["cohort"] == "TCGA-BRCA") & (prim["module"] == module)]
        b = prim[(prim["cohort"] == "METABRIC") & (prim["module"] == module)]
        if a.empty or b.empty or pd.isna(a["hr_per_sd"].iloc[0]) or pd.isna(b["hr_per_sd"].iloc[0]):
            continue
        a, b = a.iloc[0], b.iloc[0]
        out.append(dict(module=module,
                        tcga_hr=a["hr_per_sd"], tcga_lo=a["ci_lo"], tcga_hi=a["ci_hi"], tcga_p=a["p"],
                        metabric_hr=b["hr_per_sd"], metabric_lo=b["ci_lo"], metabric_hi=b["ci_hi"], metabric_p=b["p"],
                        same_direction=bool((a["hr_per_sd"] - 1) * (b["hr_per_sd"] - 1) > 0),
                        metabric_ci_excludes_1=bool(b["ci_lo"] > 1 or b["ci_hi"] < 1)))
    table = pd.DataFrame(out)
    if not table.empty:
        table["replicated"] = table["same_direction"] & table["metabric_ci_excludes_1"]
    return table

## 8. Figures

Colours are the Okabe-Ito colour-blind-safe pair (blue and vermilion).

In [ ]:
COL_LOW, COL_HIGH = "#0072B2", "#D55E00"
plt.rcParams.update({"font.size": 10, "axes.spines.top": False, "axes.spines.right": False})


def save_fig(fig, path):
    if USE_DEMO_DATA:
        fig.text(0.5, 0.5, "SIMULATED DEMO DATA\nNOT REAL RESULTS", fontsize=24, color="red",
                 alpha=0.18, ha="center", va="center", rotation=25)
    fig.savefig(path, dpi=200, bbox_inches="tight")
    plt.close(fig)


def km_figure(df, cohort, endpoint, module, path):
    d = df[["time", "event", module]].dropna()
    high = d[module] > d[module].median()
    fig, ax = plt.subplots(figsize=(6.4, 6.2))
    k_lo = KaplanMeierFitter().fit(d.loc[~high, "time"], d.loc[~high, "event"],
                                   label=f"Low score (n={int((~high).sum())})")
    k_hi = KaplanMeierFitter().fit(d.loc[high, "time"], d.loc[high, "event"],
                                   label=f"High score (n={int(high.sum())})")
    k_lo.plot_survival_function(ax=ax, color=COL_LOW, ci_alpha=0.12)
    k_hi.plot_survival_function(ax=ax, color=COL_HIGH, ci_alpha=0.12)
    lr = logrank_test(d.loc[high, "time"], d.loc[~high, "time"], d.loc[high, "event"], d.loc[~high, "event"])
    ax.set_title(f"{cohort}: {MODULE_LABELS[module]} score and {ENDPOINT_LABELS[endpoint]}\n"
                 f"median split, log-rank p = {fmt_p(lr.p_value)}", fontsize=11)
    ax.set_xlabel("Months from diagnosis")
    ax.set_ylabel("Survival probability")
    ax.set_xlim(0, MAX_MONTHS)
    ax.set_ylim(0, 1.02)
    ax.legend(loc="lower left", frameon=False)
    add_at_risk_counts(k_lo, k_hi, ax=ax, xticks=np.arange(0, MAX_MONTHS + 1, 24), rows_to_show=["At risk"])
    fig.tight_layout()
    save_fig(fig, path)


def log_axis(ax):
    """Log-scaled hazard-ratio axis with plain-number tick labels (0.5, 1, 2 ...)."""
    from matplotlib.ticker import FixedLocator, FuncFormatter, NullLocator
    ax.set_xscale("log")
    lo, hi = ax.get_xlim()
    ticks = [t for t in (0.25, 0.33, 0.5, 0.67, 0.8, 1, 1.25, 1.5, 2, 3, 4) if lo * 0.9 <= t <= hi * 1.1]
    ax.xaxis.set_major_locator(FixedLocator(ticks))
    ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:g}"))
    ax.xaxis.set_minor_locator(NullLocator())


def forest_plot(res, path):
    d = res.dropna(subset=["hr_per_sd"]).copy()
    d["order_ep"] = d["endpoint"].map({"OS": 0, "PFS": 1, "RFS": 1, "DSS": 2})
    d["order_mod"] = d["module"].map({m: i for i, m in enumerate(MODULES)})
    d["order_co"] = d["cohort"].map({"TCGA-BRCA": 0, "METABRIC": 1})
    d = d.sort_values(["order_ep", "order_mod", "order_co"]).reset_index(drop=True)
    labels = d["endpoint"] + " | " + d["module"].map(MODULE_LABELS) + " | " + d["cohort"]
    y = np.arange(len(d))[::-1].astype(float)
    fig, ax = plt.subplots(figsize=(8.6, 0.42 * len(d) + 1.8))
    ax.errorbar(d["hr_per_sd"], y + 0.13, xerr=[d["hr_per_sd"] - d["ci_lo"], d["ci_hi"] - d["hr_per_sd"]],
                fmt="o", color=COL_LOW, capsize=2, label="Unadjusted")
    if "adj_hr" in d:
        ax.errorbar(d["adj_hr"], y - 0.13, xerr=[d["adj_hr"] - d["adj_lo"], d["adj_hi"] - d["adj_hr"]],
                    fmt="s", color=COL_HIGH, capsize=2, label="Adjusted (age, stage, subtype)")
    ax.axvline(1, ls="--", color="grey", lw=1)
    log_axis(ax)
    ax.set_yticks(y)
    ax.set_yticklabels(labels)
    ax.set_xlabel("Hazard ratio per 1 SD of score (95% CI); above 1 = higher risk")
    ax.set_title("Signature scores and survival: all cohorts, endpoints and modules")
    ax.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.09), ncol=2)
    fig.tight_layout()
    save_fig(fig, path)


def subtype_forest(sub, path):
    d = sub.dropna(subset=["hr_per_sd"]).copy()
    if d.empty:
        return False
    d["order"] = d["subtype"].map({"LumA": 0, "LumB": 1, "Her2": 2, "Basal": 3})
    d = d.sort_values(["module", "order", "cohort"]).reset_index(drop=True)
    labels = d["module"].map(MODULE_LABELS) + " | " + d["subtype"] + " | " + d["cohort"] + " (events=" + d["events"].astype(str) + ")"
    y = np.arange(len(d))[::-1]
    fig, ax = plt.subplots(figsize=(8.6, 0.38 * len(d) + 1.8))
    ax.errorbar(d["hr_per_sd"], y, xerr=[d["hr_per_sd"] - d["ci_lo"], d["ci_hi"] - d["hr_per_sd"]],
                fmt="o", color=COL_LOW, capsize=2)
    ax.axvline(1, ls="--", color="grey", lw=1)
    log_axis(ax)
    ax.set_yticks(y)
    ax.set_yticklabels(labels)
    ax.set_xlabel("Hazard ratio per 1 SD, age-adjusted (95% CI)")
    ax.set_title(f"Exploratory: {ENDPOINT_LABELS[PRIMARY_ENDPOINT]} within molecular subtypes")
    fig.tight_layout()
    save_fig(fig, path)
    return True


def correlation_figure(cohorts, signature_final, path):
    genes = [g for gs in signature_final.values() for g in gs]
    fig, axes = plt.subplots(1, len(cohorts), figsize=(5.8 * len(cohorts) + 1.2, 5.4), constrained_layout=True)
    axes = np.atleast_1d(axes)
    for ax, (name, c) in zip(axes, cohorts.items()):
        corr = c["z"][genes].corr()
        im = ax.imshow(corr.to_numpy(), vmin=-1, vmax=1, cmap="coolwarm")
        ax.set_xticks(range(len(genes)))
        ax.set_xticklabels(genes, rotation=90)
        ax.set_yticks(range(len(genes)))
        ax.set_yticklabels(genes)
        cut = len(signature_final["immune_evasion"]) - 0.5
        ax.axhline(cut, color="black", lw=0.8)
        ax.axvline(cut, color="black", lw=0.8)
        ax.set_title(f"{name}: gene-gene correlation")
    fig.colorbar(im, ax=axes.tolist(), shrink=0.8, label="Pearson r")
    save_fig(fig, path)

## 9. Run everything

In [ ]:
def run_pipeline():
    cohorts = {name: build_cohort(name, cfg) for name, cfg in COHORTS.items()}
    signature_final, gene_report = finalise_signature(cohorts)
    print("\nGenes used:", {m: g for m, g in signature_final.items()})

    surv_rows, sub_rows, frames = [], [], {}
    for name, c in cohorts.items():
        c["scores"], c["z"] = compute_scores(c["expr"], signature_final)
        for endpoint in c["endpoints"]:
            df = endpoint_frame(c["tidy"], c["scores"], endpoint)
            frames[(name, endpoint)] = df
            surv_rows += analyse_frame(name, endpoint, df)
            if endpoint == PRIMARY_ENDPOINT:
                sub_rows += analyse_subtypes(name, endpoint, df)

    results = pd.DataFrame(surv_rows)
    results["p_bh"] = benjamini_hochberg(results["p"])
    if "adj_p" in results:
        results["adj_p_bh"] = benjamini_hochberg(results["adj_p"])
    sub_results = pd.DataFrame(sub_rows)
    replication = replication_table(results)

    # ---- tables
    flow = pd.DataFrame([{"cohort": n, "step": s, "n": k} for n, c in cohorts.items() for s, k in c["flow"]])
    flow.to_csv(RESULTS_DIR / "sample_flow.csv", index=False)
    gene_report.to_csv(RESULTS_DIR / "signature_genes.csv", index=False)
    results.to_csv(RESULTS_DIR / "survival_results.csv", index=False)
    sub_results.to_csv(RESULTS_DIR / "subtype_results.csv", index=False)
    replication.to_csv(RESULTS_DIR / "replication_primary.csv", index=False)

    # ---- figures
    for name in cohorts:
        for module in PRIMARY_MODULES:
            km_figure(frames[(name, PRIMARY_ENDPOINT)], name, PRIMARY_ENDPOINT, module,
                      FIG_DIR / f"km_{PRIMARY_ENDPOINT}_{module}_{name}.png")
    forest_plot(results, FIG_DIR / "forest_all_results.png")
    subtype_forest(sub_results, FIG_DIR / "forest_subtypes_exploratory.png")
    correlation_figure(cohorts, signature_final, FIG_DIR / "gene_correlation.png")

    # ---- run record
    info = dict(
        date=str(date.today()), demo_data=USE_DEMO_DATA, max_follow_up_months=MAX_MONTHS, seed=RANDOM_SEED,
        signature_requested=SIGNATURE, signature_used=signature_final,
        versions=dict(python=platform.python_version(), pandas=pd.__version__, numpy=np.__version__,
                      lifelines=lifelines.__version__, matplotlib=matplotlib.__version__, scipy=scipy.__version__),
        cohorts={n: c["meta"] for n, c in cohorts.items()})
    (RESULTS_DIR / "run_info.json").write_text(json.dumps(info, indent=2))
    return dict(cohorts=cohorts, signature=signature_final, results=results, subtypes=sub_results,
                replication=replication, flow=flow, info=info)


out = run_pipeline()

## 10. Results summary

This cell writes `results/summary.md` with the numbers exactly as computed. Nothing here is interpreted for you:
read the tables, look at the figures, and write the discussion in your own words.

In [ ]:
def df_to_md(df):
    if df.empty:
        return "_none_"
    cols = list(df.columns)
    lines = ["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)]
    for _, r in df.iterrows():
        lines.append("| " + " | ".join("" if pd.isna(v) else str(v) for v in r.tolist()) + " |")
    return "\n".join(lines)


def ci_text(hr, lo, hi):
    return "NA" if pd.isna(hr) else f"{hr:.2f} ({lo:.2f}-{hi:.2f})"


def getcol(df, name):
    """A column if it exists, otherwise a column of NaN (some models can fail or be skipped)."""
    return df[name] if name in df.columns else pd.Series(np.nan, index=df.index)


def write_summary(out):
    res = out["results"]
    prim = res[res["primary"]].copy()
    show = pd.DataFrame({
        "Cohort": prim["cohort"], "Module": prim["module"].map(MODULE_LABELS), "n": prim["n"], "Deaths": prim["events"],
        "HR per SD (95% CI)": [ci_text(a, b, c) for a, b, c in zip(getcol(prim, "hr_per_sd"), getcol(prim, "ci_lo"), getcol(prim, "ci_hi"))],
        "p": getcol(prim, "p").map(fmt_p),
        "Adjusted HR (95% CI)": [ci_text(a, b, c) for a, b, c in zip(getcol(prim, "adj_hr"), getcol(prim, "adj_lo"), getcol(prim, "adj_hi"))],
        "Adjusted p": getcol(prim, "adj_p").map(fmt_p),
        "PH test p (adj.)": getcol(prim, "ph_test_p_adjusted").map(fmt_p),
    })
    rep = out["replication"]
    rep_show = pd.DataFrame() if rep.empty else pd.DataFrame({
        "Module": rep["module"].map(MODULE_LABELS),
        "TCGA-BRCA HR (95% CI)": [ci_text(a, b, c) for a, b, c in zip(rep["tcga_hr"], rep["tcga_lo"], rep["tcga_hi"])],
        "METABRIC HR (95% CI)": [ci_text(a, b, c) for a, b, c in zip(rep["metabric_hr"], rep["metabric_lo"], rep["metabric_hi"])],
        "Same direction": rep["same_direction"], "METABRIC CI excludes 1": rep["metabric_ci_excludes_1"],
        "Replicated (rule above)": rep["replicated"]})
    flags = res[getcol(res, "ph_test_p_adjusted") < 0.05]
    lines = []
    if USE_DEMO_DATA:
        lines += ["# SIMULATED DEMO RUN: these numbers are NOT real findings", ""]
    else:
        lines += ["# Run summary (auto-generated)", ""]
    lines += [f"Date: {out['info']['date']}  |  follow-up censored at {MAX_MONTHS} months", "",
              "## Signature genes used", "",
              *[f"- **{MODULE_LABELS[m]}**: {', '.join(g)}" for m, g in out["signature"].items()], "",
              "## Sample flow", "", df_to_md(out["flow"]), "",
              f"## Primary analysis ({ENDPOINT_LABELS[PRIMARY_ENDPOINT]}, continuous score, hazard ratio per 1 SD)", "",
              df_to_md(show), "", "## Replication check (pre-specified rule)", "", df_to_md(rep_show), "",
              "## Models where proportional hazards looks doubtful (adjusted PH test p < 0.05)", "",
              df_to_md(flags[["cohort", "endpoint", "module", "ph_test_p_adjusted"]].round(4)) if len(flags) else "_none flagged_", "",
              "Full tables are in `survival_results.csv` and `subtype_results.csv`. Secondary endpoints are not identical across "
              "cohorts (progression-free survival in TCGA, relapse-free survival in METABRIC), so read them as supportive only."]
    path = RESULTS_DIR / "summary.md"
    path.write_text("\n".join(lines))
    return path


summary_path = write_summary(out)
print(summary_path.read_text())

## 11. Look at the figures

In [ ]:
try:
    from IPython.display import Image, display
    for fig_path in sorted(FIG_DIR.glob("*.png")):
        print(fig_path.name)
        display(Image(filename=str(fig_path), width=620))
except Exception as exc:                       # plain Python run, no notebook display
    print("Figures saved in", FIG_DIR, "-", exc)

## 12. Download the outputs (Colab)

In [ ]:
import zipfile

zip_path = "demo_outputs.zip" if USE_DEMO_DATA else "results_and_figures.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:      # only results and figures, never the raw data
    for folder in (RESULTS_DIR, FIG_DIR):
        for p in sorted(folder.rglob("*")):
            if p.is_file():
                zf.write(p, p.relative_to(OUT_DIR))
print("Created", zip_path)
try:
    from google.colab import files
    files.download(zip_path)
except Exception:
    pass                                       # not running in Colab

## Notes and limitations

* Both cohorts are public, retrospective and mostly from patients in Europe and North America. Nothing here says
  whether the same scores behave the same way in African patients; that question needs African cohorts.
* The signature is small and was chosen from biological reasoning, not optimised. A different, equally reasonable
  gene list could give a different answer.
* Immune-checkpoint genes partly reflect how many immune cells are in the tumour, so the immune-evasion score cannot
  separate "tumour is evading immune attack" from "tumour is heavily infiltrated".
* An association with survival is not evidence that these genes cause the outcome or that they should be treated as targets.
* TCGA has short median follow-up and few deaths, so its estimates are imprecise; METABRIC has longer follow-up
  but different treatment eras and a different expression platform.
* Overall survival in METABRIC includes deaths from other causes; the disease-specific analysis addresses this.